# 04. Quick Sort, Partition, Quickselect

Quick sort picks a **pivot**, puts smaller elements on its left and larger on its right (**partition**), then sorts both sides. Partition is the real skill: it also solves Sort Colors and Kth Largest.

**Learning order:** PARTITION → QUICK SORT → WORST CASE → QUICKSELECT → 3-WAY PARTITION → INTERVIEW THINKING

## 1. Partition (Lomuto scheme)

Pivot = last element. Pointer `i` marks the boundary of "smaller than pivot" elements.

- Scan `j` from `low` to `high - 1`.
- If `arr[j] < pivot`, swap it to position `i` and do `i += 1`.
- At the end, swap the pivot into position `i`.

After partition the pivot is in its **final sorted position**.

In [1]:
def partition(arr, low, high):
    pivot = arr[high]
    i = low
    for j in range(low, high):
        if arr[j] < pivot:
            arr[i], arr[j] = arr[j], arr[i]
            i += 1
    arr[i], arr[high] = arr[high], arr[i]
    return i

arr = [4, 7, 1, 6, 3]
p = partition(arr, 0, 4)
print(p, arr)   # 1 [1, 3, 4, 6, 7]  -> pivot 3 is at index 1

1 [1, 3, 4, 6, 7]


## 2. Quick sort

Partition, then recurse on the left part and the right part. The pivot is already placed, so it is skipped.

Choosing a **random pivot** makes the worst case very unlikely.

In [2]:
import random

def quick_sort(arr, low=0, high=None):
    if high is None:
        high = len(arr) - 1
    if low < high:
        r = random.randint(low, high)          # random pivot
        arr[r], arr[high] = arr[high], arr[r]
        p = partition(arr, low, high)
        quick_sort(arr, low, p - 1)
        quick_sort(arr, p + 1, high)
    return arr

print(quick_sort([10, 7, 8, 9, 1, 5]))   # [1, 5, 7, 8, 9, 10]

[1, 5, 7, 8, 9, 10]


## 3. Complexity and the worst case

- **Average / best:** pivot splits roughly in half → log n levels × n work = **O(n log n)**.
- **Worst:** pivot is always the smallest or largest (e.g. sorted array with last-element pivot) → n levels × n work = **O(n²)**.
- **Space:** O(log n) recursion stack on average (O(n) worst).
- **In-place, not stable.**
- Lomuto with `<` also degrades when **many elements are equal**. Fix: 3-way partition (section 5).

| | Merge sort | Quick sort |
|---|---|---|
| Worst case | O(n log n) | O(n²) (rare with random pivot) |
| Extra space | O(n) | O(log n) |
| Stable | Yes | No |
| Practical speed | good | usually faster (cache friendly) |

## 4. Quickselect: Kth element without full sorting

After a partition, the pivot is at its final index `p`.
- `p == k` → answer found.
- `p < k` → answer is on the right.
- `p > k` → answer is on the left.

We only go into **one** side, so average time is n + n/2 + n/4 + … = **O(n)** (worst O(n²), avoided mostly by random pivot).

Kth **largest** = element at index `len(arr) - k` in sorted order.

In [3]:
def find_kth_largest(nums, k):
    nums = nums[:]                    # do not change the caller's list
    target = len(nums) - k
    low, high = 0, len(nums) - 1
    while low <= high:
        r = random.randint(low, high)
        nums[r], nums[high] = nums[high], nums[r]
        p = partition(nums, low, high)
        if p == target:
            return nums[p]
        elif p < target:
            low = p + 1
        else:
            high = p - 1

print(find_kth_largest([3, 2, 1, 5, 6, 4], 2))   # 5

5


## 5. 3-way partition (Dutch National Flag)

For values with only a few distinct kinds (like 0, 1, 2 in *Sort Colors*) split into three zones: `< pivot`, `== pivot`, `> pivot`.

Pointers: `low` (end of 0s), `mid` (current), `high` (start of 2s).
- `0` → swap with `low`, move both `low` and `mid`.
- `1` → just move `mid`.
- `2` → swap with `high`, move only `high` (the swapped-in element is still unchecked).

In [4]:
def sort_colors(nums):
    low = mid = 0
    high = len(nums) - 1
    while mid <= high:
        if nums[mid] == 0:
            nums[low], nums[mid] = nums[mid], nums[low]
            low += 1
            mid += 1
        elif nums[mid] == 1:
            mid += 1
        else:
            nums[mid], nums[high] = nums[high], nums[mid]
            high -= 1
    return nums

print(sort_colors([2, 0, 2, 1, 1, 0]))   # [0, 0, 1, 1, 2, 2]

[0, 0, 1, 1, 2, 2]


## 6. Interview thinking

**Signals:** "Kth largest/smallest", "median", "sort 0/1/2 in one pass", "partition around a value", "move all X to one side".

**Kth largest — three options to mention:** sort O(n log n); min-heap of size k O(n log k); quickselect O(n) average. Pick one, justify, mention the others.

**Common mistakes:** forgetting the pivot is already in place (`p - 1`, `p + 1`), wrong target index for Kth largest, advancing `mid` after swapping with `high`, using sorted input with a fixed pivot.

## Quick Revision

- Partition puts pivot at its final index; smaller left, bigger right.
- Quick sort: average O(n log n), worst O(n²), in-place, not stable. Use random pivot.
- Quickselect: go to one side only → O(n) average. Kth largest = index `n - k`.
- Dutch flag: `low / mid / high`; after swapping with `high`, do **not** move `mid`.
- Merge sort = guaranteed + stable; quick sort = in-place + fast.